In [4]:
import polars as pl
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
import os
from datetime import datetime

### **Đặc trưng:** Phân khúc loại sản phẩm (category_l1) theo mức giá: Bình dân - Trung cấp - Cao cấp

In [3]:
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.item_chunk_0.parquet"
df_item = pl.read_parquet(path_item)
print(df_item.shape)
print(df_item.head())


(27323, 11)
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ gender_ta ┆ descripti ┆ brand_fin ┆ age_grou │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ rget_fina ┆ on_final  ┆ al        ┆ p_final  │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ l         ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ ---       ┆ str       ┆ str       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Không xác ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M    │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ định      ┆ sản phẩm  ┆ s         ┆          │
│           ┆           ┆           ┆           ┆   ┆           

In [4]:
def cluster_price_group(prices):
    prices_reshaped = prices.to_numpy().reshape(-1, 1)

    # Gom cụm 3 cluster
    kmeans = KMeans(n_clusters=3, random_state=42)
    labels = kmeans.fit_predict(prices_reshaped)

    # Tính mean của từng cluster bằng pandas
    df_temp = pd.DataFrame({"price": prices.values, "cluster": labels})
    cluster_means = df_temp.groupby("cluster")["price"].mean()

    # Sắp xếp cluster theo giá tăng dần → 0,1,2
    sorted_clusters = cluster_means.sort_values().index.tolist()

    # Mapping cluster gốc → 0,1,2
    cluster_map = {sorted_clusters[i]: i for i in range(3)}

    # Trả về danh sách nhãn theo thứ tự index
    return [cluster_map[c] for c in labels]

In [5]:
pdf = df_item.select(["item_id", "price", "category_l1"]).to_pandas()

# Tạo cột rỗng
pdf["price_segment"] = -1

for cat, group in pdf.groupby("category_l1"):
    X = group["price"].values.reshape(-1,1)

    if len(group) < 3:
        # Không gom cụm nếu số lượng ít → gán 0 hết
        pdf.loc[group.index, "price_segment"] = 0
        continue

    # Gom cụm
    kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X)

    # Remap theo thứ tự giá trung bình
    cluster_mean = pd.DataFrame({
        "cluster": labels,
        "price": group["price"].values
    }).groupby("cluster")["price"].mean().sort_values()

    mapping = {cluster: rank for rank, cluster in enumerate(cluster_mean.index)}

    # Gán nhãn theo index
    pdf.loc[group.index, "price_segment"] = [mapping[c] for c in labels]


In [6]:
print(pdf.groupby(["category_l1", "price_segment"]).size())


category_l1             price_segment
Babycare                0                1795
                        1                 165
                        2                  34
Gói Hội Viên            0                   3
                        1                   1
                        2                   3
Hóa mỹ phẩm cho bé      0                 168
                        1                 148
                        2                  29
Hóa mỹ phẩm gia đình    0                 178
                        1                 183
                        2                  26
Phụ kiện                0                1492
                        1                1134
                        2                 521
Sữa                     0                 159
                        1                 205
                        2                  73
Sữa nước                0                 131
                        1                  27
                        2                 

Mapping:
- 0: Bình dân
- 1: Trung cấp
- 2: Cao cấp

In [7]:
segment_stats = (
    pdf.groupby(["category_l1", "price_segment"])["price"]
       .agg(["min", "max", "mean", "median", "count"])
       .sort_values(["category_l1", "price_segment"])
)

print("\n=== BẢNG THỐNG KÊ THEO SEGMENT ===")
print(segment_stats)



=== BẢNG THỐNG KÊ THEO SEGMENT ===
                                               min            max  \
category_l1            price_segment                                
Babycare               0                 1000.0000   1767273.0000   
                       1              1821000.0000   6775000.0000   
                       2              6950000.0000  20990000.0000   
Gói Hội Viên           0                19000.0000     99000.0000   
                       1               169000.0000    169000.0000   
                       2               230000.0000    299000.0000   
Hóa mỹ phẩm cho bé     0                20000.0000    140000.0000   
                       1               145000.0000    290000.0000   
                       2               295000.0000    685000.0000   
Hóa mỹ phẩm gia đình   0                12000.0000    135000.0000   
                       1               139000.0000    275000.0000   
                       2               288000.0000    750000.0000  

In [8]:
df_price_seg = pl.from_pandas(pdf)


In [9]:
df_item = df_item.join(
    df_price_seg.select(["item_id", "price_segment"]),
    on="item_id",
    how="left"
)


In [10]:
print(df_item.head())
print(df_item.select("price_segment").unique())


shape: (5, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ descripti ┆ brand_fin ┆ age_group ┆ price_se │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ on_final  ┆ al        ┆ _final    ┆ gment    │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M     ┆ 0        │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ sản phẩm  ┆ s         ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ …         ┆           ┆           ┆          │
│ 001029004 ┆ 69000.000 ┆ Thời      ┆ Cơ cấu    ┆ … ┆ Không xác ┆ Con Cưng  

In [12]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\sale_pers.item_chunk_0.parquet"

df_item.write_parquet(output_path)

print(f"Đã lưu thành công vào:\n{output_path}")


Đã lưu thành công vào:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\sale_pers.item_chunk_0.parquet


**Nhận Xét:**

**1. Các nhóm ngành có phân khúc giá rất rõ ràng và hợp lý**

Những category_l1 như Sữa, Tã, Textile, Đồ chơi & Sách, TPCN, Hóa mỹ phẩm, Babycare thể hiện phân tách giá rất mạnh:
- Cụm 0 (Bình dân): giá thấp – trung bình
- Cụm 1 (Trung cấp): giá tăng đáng kể
- Cụm 2 (Cao cấp): giá cao vượt trội và rất đặc trưng

**2. Các ngành hàng giá thấp như “Phụ kiện”, “Thực phẩm cho bé”, “Vệ sinh” không quá chênh lệch giữa 0 - 1 - 2**

=> Phân bố không quá rõ ràng để ứng dụng phân cụm

**3. Chỉ có "Tã" và "Sữa" là có phân khúc Trung cấp nhiều hơn Cao cấp**

In [305]:
#===================================

### **Đặc trưng:** Số loại sản phẩm (category_l1) trung bình cho một lần mua hàng (một ngày), và phân cụm số loại giao dịch trung bình đó, gán vào 3 nhãn "Mua ít", "Mua vừa" và "Mua nhiều".

In [43]:
path_tx_0 = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet"

df_tx0 = pl.read_parquet(path_tx_0)

print("Số dòng:", df_tx0.height)
print("Số cột:", df_tx0.width)
print("\nDanh sách cột:")
print(df_tx0.columns)


Số dòng: 487850
Số cột: 13

Danh sách cột:
['item_id', 'price', 'quantity', 'customer_id', 'created_date', 'channel', 'payment', 'location', 'discount', 'category_l1', 'list_price', 'category_l2', 'discount_rate']


In [44]:
df_tx0.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,category_l1,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]",str,"decimal[38,4]",str,"decimal[38,4]"
"""7115000000004""",49000.0000,1,5254214,2024-12-24,"""In-Store""","""VietQR""",656,0.0000,"""Thực phẩm cho bé""",49000.0000,"""Snack ăn dặm""",0.0000
"""0029130000030""",69000.0000,1,7573232,2024-12-24,"""In-Store""","""Tiền mặt""",143,0.0000,"""Thực phẩm cho bé""",74000.0000,"""Bột ăn dặm""",0.0676
"""3496000000053""",75000.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,0.0000,"""Thời trang""",75000.0000,"""Quần áo & Phụ kiện sơ sinh""",0.0000
"""2700000000002""",58500.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,13000.0000,"""Vệ sinh""",65000.0000,"""Khăn khô""",0.1000
"""0029110000036""",89000.0000,1,6931560,2024-12-28,"""Android""","""MoMo""",590,10000.0000,"""Thực phẩm cho bé""",99000.0000,"""Snack ăn dặm""",0.1010


In [58]:
TX_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data"
TX_PATTERN = "sale_pers.purchase_history_daily_chunk_{}.parquet"

ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.item_chunk_0.parquet"

OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_behavior_202401_202501.parquet"

In [46]:
START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2025, 1, 31)

In [47]:
# 3. READ ITEM
df_item = pl.read_parquet(ITEM_PATH).select(
    ["item_id", "category_l1"]
)

print("Item shape:", df_item.shape)


Item shape: (27323, 2)


In [48]:
# 4. PROCESS TRANSACTION CHUNKS

customer_agg_chunks = []

for i in range(80):  # 0 → 79
    tx_path = os.path.join(TX_DIR, TX_PATTERN.format(i))
    if not os.path.exists(tx_path):
        continue

    print(f"\n=== Đang xử lý chunk {i} ===")

    tx_chunk = pl.read_parquet(
        tx_path,
        columns=["item_id", "customer_id", "created_date"]
    )

    # --- Lọc theo thời gian ---
    tx_chunk = tx_chunk.filter(
        (pl.col("created_date") >= START_DATE) &
        (pl.col("created_date") <= END_DATE)
    )

    if tx_chunk.is_empty():
        print("  -> Không có dữ liệu trong window thời gian, skip.")
        continue

    # --- Join lấy category_l1 ---
    tx_joined = tx_chunk.join(
        df_item,
        on="item_id",
        how="left"
    )

    # --- Đếm số category_l1 khác nhau mỗi ngày ---
    daily_cat_count = (
        tx_joined
        .group_by(["customer_id", "created_date"])
        .agg(
            pl.col("category_l1").n_unique().alias("unique_categories")
        )
    )

    # --- Aggregate trong chunk ---
    cust_chunk = (
        daily_cat_count
        .group_by("customer_id")
        .agg([
            pl.col("unique_categories").sum().alias("sum_unique_categories"),
            pl.len().alias("num_days")
        ])
    )

    customer_agg_chunks.append(cust_chunk)

print("\n=== Hoàn thành xử lý tất cả transaction chunks ===")



=== Đang xử lý chunk 0 ===

=== Đang xử lý chunk 1 ===

=== Đang xử lý chunk 2 ===

=== Đang xử lý chunk 3 ===

=== Đang xử lý chunk 4 ===

=== Đang xử lý chunk 5 ===

=== Đang xử lý chunk 6 ===

=== Đang xử lý chunk 7 ===

=== Đang xử lý chunk 8 ===

=== Đang xử lý chunk 9 ===

=== Đang xử lý chunk 10 ===

=== Đang xử lý chunk 11 ===

=== Đang xử lý chunk 12 ===

=== Đang xử lý chunk 13 ===

=== Đang xử lý chunk 14 ===

=== Đang xử lý chunk 15 ===

=== Đang xử lý chunk 16 ===

=== Đang xử lý chunk 17 ===

=== Đang xử lý chunk 18 ===

=== Đang xử lý chunk 19 ===

=== Đang xử lý chunk 20 ===

=== Đang xử lý chunk 21 ===

=== Đang xử lý chunk 22 ===

=== Đang xử lý chunk 23 ===

=== Đang xử lý chunk 24 ===

=== Đang xử lý chunk 25 ===

=== Đang xử lý chunk 26 ===

=== Đang xử lý chunk 27 ===

=== Đang xử lý chunk 28 ===

=== Đang xử lý chunk 29 ===

=== Đang xử lý chunk 30 ===

=== Đang xử lý chunk 31 ===

=== Đang xử lý chunk 32 ===

=== Đang xử lý chunk 33 ===

=== Đang xử lý chunk 34

In [49]:
# 5. AGG FINAL PER CUSTOMER

customer_agg_all = pl.concat(customer_agg_chunks, how="vertical")

customer_final = (
    customer_agg_all
    .group_by("customer_id")
    .agg([
        pl.col("sum_unique_categories").sum().alias("total_unique_categories"),
        pl.col("num_days").sum().alias("total_days")
    ])
    .with_columns(
        (pl.col("total_unique_categories") / pl.col("total_days"))
        .alias("avg_categories_per_day")
    )
)

print("Số khách hàng:", customer_final.height)

Số khách hàng: 2569978


In [50]:
# 6. KMEANS CLUSTERING (3 SEGMENTS)
X = customer_final["avg_categories_per_day"].to_numpy().reshape(-1, 1)

kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X)

customer_final = customer_final.with_columns(
    pl.Series("buy_segment_raw", labels)
)

# --- Remap cluster theo thứ tự centroid ---
cluster_stats = (
    customer_final
    .group_by("buy_segment_raw")
    .agg(pl.col("avg_categories_per_day").mean().alias("mean_avg"))
    .sort("mean_avg")
)

mapping = {
    row["buy_segment_raw"]: idx
    for idx, row in enumerate(cluster_stats.iter_rows(named=True))
}

customer_final = customer_final.with_columns(
    pl.col("buy_segment_raw").replace(mapping).alias("buy_segment")
)


In [51]:
# Bảng tổng hợp theo segment

segment_summary = (
    customer_final
    .group_by("buy_segment")
    .agg([
        pl.count().alias("num_customers"),
        pl.col("avg_categories_per_day").min().alias("min_avg"),
        pl.col("avg_categories_per_day").median().alias("median_avg"),
        pl.col("avg_categories_per_day").mean().alias("mean_avg"),
        pl.col("avg_categories_per_day").max().alias("max_avg"),
    ])
    .sort("buy_segment")
)

print("=== SUMMARY THEO SEGMENT ===")
print(segment_summary)


=== SUMMARY THEO SEGMENT ===
shape: (3, 6)
┌─────────────┬───────────────┬──────────┬────────────┬──────────┬──────────┐
│ buy_segment ┆ num_customers ┆ min_avg  ┆ median_avg ┆ mean_avg ┆ max_avg  │
│ ---         ┆ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---      │
│ i32         ┆ u32           ┆ f64      ┆ f64        ┆ f64      ┆ f64      │
╞═════════════╪═══════════════╪══════════╪════════════╪══════════╪══════════╡
│ 0           ┆ 1663256       ┆ 1.0      ┆ 1.0        ┆ 1.069234 ┆ 1.421569 │
│ 1           ┆ 778872        ┆ 1.421687 ┆ 1.75       ┆ 1.775453 ┆ 2.495413 │
│ 2           ┆ 127850        ┆ 2.5      ┆ 3.0        ┆ 3.219877 ┆ 11.0     │
└─────────────┴───────────────┴──────────┴────────────┴──────────┴──────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_500\1147667939.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_customers"),


In [52]:
# In mẫu khách hàng mỗi segment

for seg in [0, 1, 2]:
    print(f"\n=== SAMPLE SEGMENT {seg} ===")
    print(
        customer_final
        .filter(pl.col("buy_segment") == seg)
        .select(["customer_id", "avg_categories_per_day"])
        .sort("avg_categories_per_day")
        .head(10)
    )



=== SAMPLE SEGMENT 0 ===
shape: (10, 2)
┌─────────────┬────────────────────────┐
│ customer_id ┆ avg_categories_per_day │
│ ---         ┆ ---                    │
│ i32         ┆ f64                    │
╞═════════════╪════════════════════════╡
│ 4610623     ┆ 1.0                    │
│ 7225148     ┆ 1.0                    │
│ 8276276     ┆ 1.0                    │
│ 7905693     ┆ 1.0                    │
│ 7141629     ┆ 1.0                    │
│ 7384528     ┆ 1.0                    │
│ 8291344     ┆ 1.0                    │
│ 7407724     ┆ 1.0                    │
│ 8008067     ┆ 1.0                    │
│ 7121720     ┆ 1.0                    │
└─────────────┴────────────────────────┘

=== SAMPLE SEGMENT 1 ===
shape: (10, 2)
┌─────────────┬────────────────────────┐
│ customer_id ┆ avg_categories_per_day │
│ ---         ┆ ---                    │
│ i32         ┆ f64                    │
╞═════════════╪════════════════════════╡
│ 6374055     ┆ 1.421687               │
│ 3303190     ┆ 

In [53]:
segment_counts = (
    customer_final
    .group_by("buy_segment")
    .agg(
        pl.count().alias("num_customers")
    )
    .sort("buy_segment")
)

print("=== SỐ LƯỢNG KHÁCH HÀNG THEO SEGMENT ===")
print(segment_counts)


=== SỐ LƯỢNG KHÁCH HÀNG THEO SEGMENT ===
shape: (3, 2)
┌─────────────┬───────────────┐
│ buy_segment ┆ num_customers │
│ ---         ┆ ---           │
│ i32         ┆ u32           │
╞═════════════╪═══════════════╡
│ 0           ┆ 1663256       │
│ 1           ┆ 778872        │
│ 2           ┆ 127850        │
└─────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_500\3131816565.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_customers")


In [54]:
customer_final

customer_id,total_unique_categories,total_days,avg_categories_per_day,buy_segment_raw,buy_segment
i32,u32,u32,f64,i32,i32
4610623,1,1,1.0,0,0
7225148,1,1,1.0,0,0
8276276,1,1,1.0,0,0
7905693,1,1,1.0,0,0
7144273,3,2,1.5,2,1
…,…,…,…,…,…
7366664,9,5,1.8,2,1
1391920,1,1,1.0,0,0
4511223,3,1,3.0,1,2


In [55]:
customer_final = customer_final.drop("buy_segment_raw")


In [56]:
customer_final

customer_id,total_unique_categories,total_days,avg_categories_per_day,buy_segment
i32,u32,u32,f64,i32
4610623,1,1,1.0,0
7225148,1,1,1.0,0
8276276,1,1,1.0,0
7905693,1,1,1.0,0
7144273,3,2,1.5,1
…,…,…,…,…
7366664,9,5,1.8,1
1391920,1,1,1.0,0
4511223,3,1,3.0,2


In [59]:
# 7. SAVE
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
customer_final.select(
    ["customer_id", "avg_categories_per_day", "buy_segment"]
).write_parquet(OUTPUT_PATH)

print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_behavior_202401_202501.parquet


### **Đặc trưng:** Mức độ cao cấp của khách hàng - dựa trên "Sữa", "Tã"
- Bước 1: Gom nhóm giá các mặt hàng có category_l1 = 'sữa' theo các nhãn "Bình dân", "Trung cấp" và "cao cấp".

- Bước 2: Thống kê xem khách hàng đó mua sữa thuộc nhóm nào nhiều nhất để gán nhãn cho khách hàng đó.

In [60]:
# 1. PATHS (PHASE 3)

ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\sale_pers.item_chunk_0.parquet"

TX_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data"
TX_PATTERN = "sale_pers.purchase_history_daily_chunk_{}.parquet"

OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_luxury_202401_202501.parquet"


In [61]:
# 2. TIME WINDOW

START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2025, 1, 31)


In [62]:
# 3. LOAD ITEM (CHỈ GIỮ CẦN THIẾT)

df_item = pl.read_parquet(
    ITEM_PATH,
    columns=["item_id", "category_l1", "price_segment"]
)

print("ITEM shape:", df_item.shape)
print(df_item.select(["category_l1", "price_segment"]).unique())


ITEM shape: (27323, 3)
shape: (45, 2)
┌────────────────────────┬───────────────┐
│ category_l1            ┆ price_segment │
│ ---                    ┆ ---           │
│ str                    ┆ i64           │
╞════════════════════════╪═══════════════╡
│ Babycare               ┆ 1             │
│ Tã                     ┆ 0             │
│ Textile                ┆ 2             │
│ Đồ chơi & Sách         ┆ 1             │
│ Textile                ┆ 1             │
│ …                      ┆ …             │
│ Thực phẩm cho gia đình ┆ 1             │
│ Hóa mỹ phẩm gia đình   ┆ 1             │
│ Thực phẩm cho bé       ┆ 1             │
│ Đồ chơi & Sách         ┆ 0             │
│ Vệ sinh                ┆ 2             │
└────────────────────────┴───────────────┘


In [63]:
# 4. PROCESS TRANSACTION CHUNKS

important_cats = ["Sữa", "Tã"]
customer_seg_chunks = []

for i in range(80):  # 0 → 79
    tx_path = os.path.join(TX_DIR, TX_PATTERN.format(i))
    if not os.path.exists(tx_path):
        continue

    print(f"\n=== Processing chunk {i} ===")

    df_tx = pl.read_parquet(
        tx_path,
        columns=["customer_id", "item_id", "created_date"]
    )

    # --- Lọc thời gian ---
    df_tx = df_tx.filter(
        (pl.col("created_date") >= START_DATE) &
        (pl.col("created_date") <= END_DATE)
    )

    if df_tx.is_empty():
        print("  -> No data in time window, skip")
        continue

    # --- Join item ---
    df_join = df_tx.join(df_item, on="item_id", how="left")

    # --- Lọc Sữa + Tã ---
    df_filtered = df_join.filter(
        pl.col("category_l1").is_in(important_cats)
    )

    if df_filtered.is_empty():
        print("  -> No Sữa / Tã in this chunk")
        continue

    # --- Count purchase theo price_segment ---
    cust_seg = (
        df_filtered
        .group_by(["customer_id", "price_segment"])
        .agg(pl.count().alias("purchase_count"))
    )

    customer_seg_chunks.append(cust_seg)

print("\n=== Done reading all chunks ===")



=== Processing chunk 0 ===

=== Processing chunk 1 ===

=== Processing chunk 2 ===


C:\Users\PC\AppData\Local\Temp\ipykernel_500\1162869475.py:44: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("purchase_count"))



=== Processing chunk 3 ===

=== Processing chunk 4 ===

=== Processing chunk 5 ===

=== Processing chunk 6 ===

=== Processing chunk 7 ===

=== Processing chunk 8 ===

=== Processing chunk 9 ===

=== Processing chunk 10 ===

=== Processing chunk 11 ===

=== Processing chunk 12 ===

=== Processing chunk 13 ===

=== Processing chunk 14 ===

=== Processing chunk 15 ===

=== Processing chunk 16 ===

=== Processing chunk 17 ===

=== Processing chunk 18 ===

=== Processing chunk 19 ===

=== Processing chunk 20 ===

=== Processing chunk 21 ===

=== Processing chunk 22 ===

=== Processing chunk 23 ===

=== Processing chunk 24 ===

=== Processing chunk 25 ===

=== Processing chunk 26 ===

=== Processing chunk 27 ===

=== Processing chunk 28 ===

=== Processing chunk 29 ===

=== Processing chunk 30 ===

=== Processing chunk 31 ===

=== Processing chunk 32 ===

=== Processing chunk 33 ===

=== Processing chunk 34 ===

=== Processing chunk 35 ===

=== Processing chunk 36 ===

=== Processing chunk

In [64]:
# 5. AGG FINAL PER CUSTOMER

customer_seg_all = pl.concat(customer_seg_chunks, how="vertical")

customer_seg_all = (
    customer_seg_all
    .group_by(["customer_id", "price_segment"])
    .agg(pl.col("purchase_count").sum())
)

print("\n=== SAMPLE customer x price_segment ===")
print(customer_seg_all.head(10))


=== SAMPLE customer x price_segment ===
shape: (10, 3)
┌─────────────┬───────────────┬────────────────┐
│ customer_id ┆ price_segment ┆ purchase_count │
│ ---         ┆ ---           ┆ ---            │
│ i32         ┆ i64           ┆ u32            │
╞═════════════╪═══════════════╪════════════════╡
│ 6410373     ┆ 0             ┆ 3              │
│ 7402309     ┆ 1             ┆ 1              │
│ 3961256     ┆ 0             ┆ 1              │
│ 7163318     ┆ 0             ┆ 1              │
│ 2571150     ┆ 1             ┆ 1              │
│ 2507640     ┆ 1             ┆ 21             │
│ 4167887     ┆ 0             ┆ 3              │
│ 815360      ┆ 1             ┆ 6              │
│ 7712471     ┆ 1             ┆ 1              │
│ 3088089     ┆ 1             ┆ 1              │
└─────────────┴───────────────┴────────────────┘


In [65]:
# 6. PIVOT + LUXURY LEVEL

customer_pivot = (
    customer_seg_all
    .pivot(
        index="customer_id",
        columns="price_segment",
        values="purchase_count",
        aggregate_function="first"
    )
    .fill_null(0)
    .rename({
        "0": "seg_0",   # Bình dân
        "1": "seg_1",   # Trung cấp
        "2": "seg_2",   # Cao cấp
    })
)

print("\n=== PIVOT TABLE (SAMPLE) ===")
print(customer_pivot.head(10))

customer_luxury = customer_pivot.with_columns(
    pl.when((pl.col("seg_2") >= pl.col("seg_1")) & (pl.col("seg_2") >= pl.col("seg_0")))
        .then(2)
    .when((pl.col("seg_1") >= pl.col("seg_0")) & (pl.col("seg_1") >= pl.col("seg_2")))
        .then(1)
    .otherwise(0)
    .alias("luxury_level")
)

print("\n=== SAMPLE luxury_level ===")
print(customer_luxury.head(10))


C:\Users\PC\AppData\Local\Temp\ipykernel_500\4280525608.py:5: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(



=== PIVOT TABLE (SAMPLE) ===
shape: (10, 4)
┌─────────────┬───────┬───────┬───────┐
│ customer_id ┆ seg_0 ┆ seg_1 ┆ seg_2 │
│ ---         ┆ ---   ┆ ---   ┆ ---   │
│ i32         ┆ u32   ┆ u32   ┆ u32   │
╞═════════════╪═══════╪═══════╪═══════╡
│ 6410373     ┆ 3     ┆ 2     ┆ 0     │
│ 7402309     ┆ 2     ┆ 1     ┆ 0     │
│ 3961256     ┆ 1     ┆ 0     ┆ 0     │
│ 7163318     ┆ 1     ┆ 5     ┆ 0     │
│ 2571150     ┆ 10    ┆ 1     ┆ 0     │
│ 2507640     ┆ 2     ┆ 21    ┆ 1     │
│ 4167887     ┆ 3     ┆ 9     ┆ 0     │
│ 815360      ┆ 19    ┆ 6     ┆ 0     │
│ 7712471     ┆ 0     ┆ 1     ┆ 0     │
│ 3088089     ┆ 0     ┆ 1     ┆ 0     │
└─────────────┴───────┴───────┴───────┘

=== SAMPLE luxury_level ===
shape: (10, 5)
┌─────────────┬───────┬───────┬───────┬──────────────┐
│ customer_id ┆ seg_0 ┆ seg_1 ┆ seg_2 ┆ luxury_level │
│ ---         ┆ ---   ┆ ---   ┆ ---   ┆ ---          │
│ i32         ┆ u32   ┆ u32   ┆ u32   ┆ i32          │
╞═════════════╪═══════╪═══════╪═══════╪════════════

In [66]:
# 7. CHECK PHÂN BỐ

level_counts = (
    customer_luxury
    .group_by("luxury_level")
    .agg(pl.count().alias("num_customers"))
    .sort("luxury_level")
)

print("\n=== SỐ LƯỢNG CUSTOMER THEO LUXURY LEVEL ===")
print(level_counts)



=== SỐ LƯỢNG CUSTOMER THEO LUXURY LEVEL ===
shape: (3, 2)
┌──────────────┬───────────────┐
│ luxury_level ┆ num_customers │
│ ---          ┆ ---           │
│ i32          ┆ u32           │
╞══════════════╪═══════════════╡
│ 0            ┆ 419556        │
│ 1            ┆ 830821        │
│ 2            ┆ 99641         │
└──────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_500\2959444835.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [67]:
# 8. SAVE CLEAN FEATURE

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

customer_luxury.select(
    ["customer_id", "luxury_level"]
).write_parquet(OUTPUT_PATH)

print("\nĐã lưu file:", OUTPUT_PATH)


Đã lưu file: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_luxury_202401_202501.parquet


In [ ]:
#==================================================

### **Đặc trưng:** Tuổi hiện tại của bé
Dựa trên age_group, step_1, step_2 và mom

In [69]:
import glob
import re
from datetime import date

In [70]:
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\sale_pers.item_chunk_0.parquet"
TRANS_DIR =  r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data"

OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_age_features_2401_2501.parquet"

PREDICTION_DATE = date(2025, 1, 31)

STEP1_AGE_MONTHS = 3.0       # Step1 dùng cho 0–6M → trung bình ~3M
AGE_0_3M_MONTHS = 1.5        # 0–3M → trung bình ~1.5M
MOM_AGE_MONTHS = 0.0         # bạn yêu cầu = 0


In [71]:
print("Đang load ITEM...")
item = pl.read_parquet(ITEM_PATH)

item = item.select([
    "item_id", 
    "category_l1", "category_l2",
    "description_final",
    "age_group_final"
])

item = item.with_columns(
    pl.col("description_final").str.to_lowercase().alias("desc_lc")
)



Đang load ITEM...


In [72]:
# FLAG Step1 (chỉ nhận khi category_l1 = Sữa)

step1_positive = [
    "step 1", "step-1", "step1",
    "stage 1", "stage-1", "stage1",
    "cho bé 0+", "cho bé 0-6", "từ 0 tháng",
    "trẻ sơ sinh", "newborn"
]

step1_negative = [
    "bước 1:"       # chắc chắn là hướng dẫn
]


item = item.with_columns([

    # Positive: chứa 1 trong các pattern
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_positive
    ]).alias("tmp_step1_pos"),

    # Negative
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_negative
    ]).alias("tmp_step1_neg"),
])

item = item.with_columns([
    (
        (pl.col("category_l1") == "Sữa") &
        pl.col("tmp_step1_pos") &
        (~pl.col("tmp_step1_neg"))
    ).alias("is_step1")
]).drop(["tmp_step1_pos", "tmp_step1_neg"])



In [73]:
item.select(
    pl.col("is_step1").value_counts()
)


is_step1
struct[2]
"{false,27261}"
"{true,62}"


In [74]:
# ====== FLAG is_age_0_6M (CHUẨN THEO ĐỊNH NGHĨA CUỐI) ======

item = item.with_columns(
    pl.col("age_group_final")
    .cast(pl.Utf8)
    .str.to_lowercase()
    .alias("age_lc")
)

# (A) RANGE giao [0,6]: min < 6, max <= 6
# Bắt các dạng: 0-3M, 1-5M, 2-6M, 3-6M, 4-6M, 5-6M
regex_range_0_6 = r"\b[0-5]\s*[-–]\s*[1-6]\s*m\b"

# (B) Dạng "Từ xM" với x ∈ {0,1,2,3}
regex_from_0_3 = r"\btừ\s*[0-3]\s*m\b"

item = item.with_columns(
    (
        pl.col("age_lc").str.contains(regex_range_0_6) |
        pl.col("age_lc").str.contains(regex_from_0_3)
    ).alias("is_age_0_3M")
)


In [75]:
item.select(
    pl.col("is_age_0_3M").value_counts()
)


is_age_0_3M
struct[2]
"{false,25839}"
"{true,1484}"


In [76]:
#FLAG is_mom (cẩn thận ĐẦM BẦU)

mom_positive = [
    #"sữa bầu",
    #"sua bau",
    #"sữa cho mẹ",
    #"dinh dưỡng cho mẹ",
    #"dành cho mẹ",
    "mom"
]


mom_negative = [
    "đầm bầu", "váy bầu", "áo bầu"
]

item = item.with_columns([
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_positive
    ]).alias("tmp_mom_pos"),

    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_negative
    ]).alias("tmp_mom_neg"),
])

item = item.with_columns([
    (
        (pl.col("tmp_mom_pos")) & 
        (~pl.col("tmp_mom_neg"))
    ).alias("is_mom")
]).drop(["tmp_mom_pos", "tmp_mom_neg"])


In [77]:
#ĐỌC 20 CHUNK TRANSACTION + JOIN ITEM + TÍNH NGÀY MIN/MAX

transaction_files = sorted(glob.glob(TRANS_DIR + r"\sale_pers.purchase_history_daily_chunk_*.parquet"))
print("Số file transaction:", len(transaction_files))

per_chunk_stats = []


Số file transaction: 80


In [78]:
for file in transaction_files:
    print("Đang xử lý:", file)

    trans = (
        pl.read_parquet(file)
        .select(["item_id", "customer_id", "created_date"])
        .with_columns(pl.col("created_date").cast(pl.Date))
    )

    # JOIN ITEM
    trans_j = trans.join(
        item.select(["item_id", "is_step1", "is_age_0_3M", "is_mom"]),
        on="item_id",
        how="left"
    )

    # AGG theo customer
    stats = (
        trans_j.group_by("customer_id")
        .agg([
            pl.col("created_date").filter(pl.col("is_step1")).min().alias("first_date_buy_step1"),
            pl.col("created_date").filter(pl.col("is_age_0_3M")).min().alias("first_date_buy_age_group_0_3M"),
            pl.col("created_date").filter(pl.col("is_mom")).max().alias("last_date_buy_milk4mom"),
        ])
    )

    per_chunk_stats.append(stats)


Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_10.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_11.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_12.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data\sale_pers.purchase_history_daily_chunk_13.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing 

In [79]:
#GỘP KẾT QUẢ CHUNK + LẤY MIN/MAX CUỐI CÙNG

print("\n=== Gộp tất cả chunk lại ===")
customer_dates_all = pl.concat(per_chunk_stats, how="vertical_relaxed")
print("Shape sau gộp:", customer_dates_all.shape)


customer_dates_final = (
    customer_dates_all
    .group_by("customer_id")
    .agg([
        pl.col("first_date_buy_step1").min().alias("first_date_buy_step1"),
        pl.col("first_date_buy_age_group_0_3M").min().alias("first_date_buy_age_group_0_3M"),
        pl.col("last_date_buy_milk4mom").max().alias("last_date_buy_milk4mom"),
    ])
)



=== Gộp tất cả chunk lại ===
Shape sau gộp: (15859985, 4)


In [80]:
print("\n=== 20 dòng mẫu có thông tin tuổi ===")
sample_nonnull = customer_dates_final.filter(
    pl.col("first_date_buy_step1").is_not_null() |
    pl.col("first_date_buy_age_group_0_3M").is_not_null() |
    pl.col("last_date_buy_milk4mom").is_not_null()
).head(20)

print(sample_nonnull)



=== 20 dòng mẫu có thông tin tuổi ===
shape: (20, 4)
┌─────────────┬──────────────────────┬───────────────────────────────┬────────────────────────┐
│ customer_id ┆ first_date_buy_step1 ┆ first_date_buy_age_group_0_3M ┆ last_date_buy_milk4mom │
│ ---         ┆ ---                  ┆ ---                           ┆ ---                    │
│ i32         ┆ date                 ┆ date                          ┆ date                   │
╞═════════════╪══════════════════════╪═══════════════════════════════╪════════════════════════╡
│ 6704093     ┆ null                 ┆ 2024-02-03                    ┆ null                   │
│ 8171285     ┆ 2025-01-14           ┆ null                          ┆ null                   │
│ 2692649     ┆ null                 ┆ 2024-05-19                    ┆ null                   │
│ 2503898     ┆ null                 ┆ 2024-06-10                    ┆ null                   │
│ 6673701     ┆ null                 ┆ 2024-11-02                    ┆ 2024-11-02 

In [81]:
# TÍNH TUỔI HIỆN TẠI (age_by_*)
pred_date_lit = pl.lit(PREDICTION_DATE).cast(pl.Date)

customer_with_age = (
    customer_dates_final
    .with_columns([
        # Lấy số ngày từ hiệu hai ngày
        (pred_date_lit - pl.col("first_date_buy_step1")).dt.total_days().alias("days_from_step1"),
        (pred_date_lit - pl.col("first_date_buy_age_group_0_3M")).dt.total_days().alias("days_from_age_group"),
        (pred_date_lit - pl.col("last_date_buy_milk4mom")).dt.total_days().alias("days_from_milk4mom"),
    ])
    .with_columns([
        (pl.col("days_from_step1") / 30).alias("months_from_step1"),
        (pl.col("days_from_age_group") / 30).alias("months_from_age_group"),
        (pl.col("days_from_milk4mom") / 30).alias("months_from_milk4mom"),
    ])
    .with_columns([
        # FINAL AGE
        (STEP1_AGE_MONTHS + pl.col("months_from_step1")).alias("age_by_step1"),
        (AGE_0_3M_MONTHS + pl.col("months_from_age_group")).alias("age_by_age_group"),
        (MOM_AGE_MONTHS + pl.col("months_from_milk4mom")).alias("age_by_milk4mom"),
    ])
    .select([
        "customer_id",
        "first_date_buy_step1", "age_by_step1",
        "first_date_buy_age_group_0_3M", "age_by_age_group",
        "last_date_buy_milk4mom", "age_by_milk4mom",
    ])
)


In [82]:
# check

print("\n=== THỐNG KÊ NON-NULL ===")

print("Có Step1:", customer_with_age.filter(pl.col("age_by_step1").is_not_null()).height)
print("Có age_group 0–3M:", customer_with_age.filter(pl.col("age_by_age_group").is_not_null()).height)
print("Có Mom:", customer_with_age.filter(pl.col("age_by_milk4mom").is_not_null()).height)



=== THỐNG KÊ NON-NULL ===
Có Step1: 337570
Có age_group 0–3M: 949335
Có Mom: 175153


In [83]:
# check

print("\n=== Mẫu khách có cả 3 tín hiệu ===")
print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null() &
        pl.col("age_by_milk4mom").is_not_null()
    )
    .head(20)
)



=== Mẫu khách có cả 3 tín hiệu ===
shape: (20, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ age_by_step ┆ first_date_ ┆ age_by_age_ ┆ last_date_b ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ 1           ┆ buy_age_gro ┆ group       ┆ uy_milk4mom ┆ 4mom        │
│ i32         ┆ ---          ┆ ---         ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         │
│             ┆ date         ┆ f64         ┆ ---         ┆ f64         ┆ date        ┆ f64         │
│             ┆              ┆             ┆ date        ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 4680482     ┆ 2024-03-30   ┆ 13.233333   ┆ 2024-01-07  ┆ 14.5        ┆ 2025-01-22  ┆ 0.3         │
│ 7416647     ┆ 2024-04-26   ┆ 12.333333   ┆ 2024-04-26  ┆ 10.833333   ┆ 2024-06-28  ┆ 7.233333    │
│ 3496055     ┆ 2024-12-19   ┆ 4.433333 

In [84]:
print("\n=== Random 20 khách có cả 3 tín hiệu ===")
print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null() &
        pl.col("age_by_milk4mom").is_not_null()
    )
    .sample(n=20, with_replacement=False)
)



=== Random 20 khách có cả 3 tín hiệu ===
shape: (20, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ age_by_step ┆ first_date_ ┆ age_by_age_ ┆ last_date_b ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ 1           ┆ buy_age_gro ┆ group       ┆ uy_milk4mom ┆ 4mom        │
│ i32         ┆ ---          ┆ ---         ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         │
│             ┆ date         ┆ f64         ┆ ---         ┆ f64         ┆ date        ┆ f64         │
│             ┆              ┆             ┆ date        ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 6202351     ┆ 2024-05-04   ┆ 12.066667   ┆ 2024-01-14  ┆ 14.266667   ┆ 2024-04-30  ┆ 9.2         │
│ 6699420     ┆ 2024-01-26   ┆ 15.366667   ┆ 2024-04-07  ┆ 11.466667   ┆ 2024-02-05  ┆ 12.033333   │
│ 3782696     ┆ 2024-01-09   ┆ 15.

 **age_by_milk4mom không tốt lắm nên không dùng nó để tạo cột tuổi trung bình cuối cùng.**

In [85]:
total_customers = customer_with_age.select(
    pl.col("customer_id").n_unique()
).item()

print("Tổng số customer:", total_customers)


Tổng số customer: 2569978


In [86]:
customers_with_baby_signal = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() |
        pl.col("age_by_age_group").is_not_null()
    )
    .select(pl.col("customer_id").n_unique())
    .item()
)

print("Customer có ít nhất 1 tín hiệu em bé:", customers_with_baby_signal)


Customer có ít nhất 1 tín hiệu em bé: 1015961


Chỉ xét có cả 2:
- <=3 tháng  : 222,030  (~87.4%)
- 3–6 tháng : 22,130   (~8.7%)
- >6 tháng  : 9,980    (~3.9%)


In [87]:
# tìm ra những khách hàng có cả age_by_step1 và age_by_age_group, tính độ lệch

print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .select(
        (pl.col("age_by_step1") - pl.col("age_by_age_group")).abs().alias("diff")
    )
    .describe()
)


shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ diff     │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 270944.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 1.94253  │
│ std        ┆ 1.725645 │
│ min        ┆ 0.0      │
│ 25%        ┆ 1.466667 │
│ 50%        ┆ 1.5      │
│ 75%        ┆ 1.6      │
│ max        ┆ 14.6     │
└────────────┴──────────┘


- Trung bình thì lệch tầm 1.7 tháng (mean = 1.67 (tháng))
- Có outlier khá mạnh (max = 10.67 tháng)

In [88]:
# Các khách hàng có độ lệch lớn nhất
large_diff = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .with_columns(
        (pl.col("age_by_step1") - pl.col("age_by_age_group"))
        .abs()
        .alias("diff")
    )
    .sort("diff", descending=True)
)

print("\n=== TOP 20 khách hàng có độ lệch tuổi lớn nhất ===")
print(
    large_diff.select([
        "customer_id",
        "first_date_buy_step1",
        "age_by_step1",
        "first_date_buy_age_group_0_3M",
        "age_by_age_group",
        "diff"
    ]).head(20)
)



=== TOP 20 khách hàng có độ lệch tuổi lớn nhất ===
shape: (20, 6)
┌─────────────┬───────────────────┬──────────────┬──────────────────┬──────────────────┬───────────┐
│ customer_id ┆ first_date_buy_st ┆ age_by_step1 ┆ first_date_buy_a ┆ age_by_age_group ┆ diff      │
│ ---         ┆ ep1               ┆ ---          ┆ ge_group_0_3M    ┆ ---              ┆ ---       │
│ i32         ┆ ---               ┆ f64          ┆ ---              ┆ f64              ┆ f64       │
│             ┆ date              ┆              ┆ date             ┆                  ┆           │
╞═════════════╪═══════════════════╪══════════════╪══════════════════╪══════════════════╪═══════════╡
│ 6714164     ┆ 2024-01-03        ┆ 16.133333    ┆ 2025-01-30       ┆ 1.533333         ┆ 14.6      │
│ 3823697     ┆ 2024-01-04        ┆ 16.1         ┆ 2025-01-30       ┆ 1.533333         ┆ 14.566667 │
│ 647928      ┆ 2024-01-02        ┆ 16.166667    ┆ 2025-01-28       ┆ 1.6              ┆ 14.566667 │
│ 6373792     ┆ 2024-01-

In [89]:
# Đếm số khách hàng có độ lệch > 6 tháng
num_diff_gt_10 = (
    large_diff
    .filter(pl.col("diff") > 5)
    .select(pl.count())
    .item()
)

print(f"\n=== Số khách hàng có |age_by_step1 - age_by_age_group| > 5 tháng ===")
print(num_diff_gt_10)



=== Số khách hàng có |age_by_step1 - age_by_age_group| > 5 tháng ===
17661


C:\Users\PC\AppData\Local\Temp\ipykernel_500\578139962.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .select(pl.count())


In [90]:
customer_with_age = customer_with_age.with_columns([
    # Tính độ lệch nếu có đủ 2 tín hiệu
    pl.when(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .then((pl.col("age_by_step1") - pl.col("age_by_age_group")).abs())
    .otherwise(None)
    .alias("age_diff")
])

customer_with_age = customer_with_age.with_columns([
    pl.when(
        # chỉ có step1
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_null()
    ).then(pl.col("age_by_step1"))

    .when(
        # chỉ có age_group
        pl.col("age_by_step1").is_null() &
        pl.col("age_by_age_group").is_not_null()
    ).then(pl.col("age_by_age_group"))

    .when(
        # lệch <= 3 tháng → trung bình
        pl.col("age_diff") <= 3
    ).then(
        (pl.col("age_by_step1") + pl.col("age_by_age_group")) / 2
    )

    .when(
        # 3 < lệch <= 6 → ưu tiên step1
        (pl.col("age_diff") > 3) & (pl.col("age_diff") <= 5)
    ).then(pl.col("age_by_step1"))

    .otherwise(None)   # lệch > 6 → loại
    .alias("age_final")
])


In [92]:
customer_with_age.head(10)


customer_id,first_date_buy_step1,age_by_step1,first_date_buy_age_group_0_3M,age_by_age_group,last_date_buy_milk4mom,age_by_milk4mom,age_diff,age_final
i32,date,f64,date,f64,date,f64,f64,f64
4748251,null,null,null,null,null,null,null,null
6704093,null,null,2024-02-03,13.6,null,null,null,13.6
8092223,null,null,null,null,null,null,null,null
7105333,null,null,null,null,null,null,null,null
2979082,null,null,null,null,null,null,null,null
4045004,null,null,null,null,null,null,null,null
8171285,2025-01-14,3.566667,null,null,null,null,null,3.566667
7577480,null,null,null,null,null,null,null,null
7582327,null,null,null,null,null,null,null,null


In [93]:
diff_bucket_stats = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .with_columns(
        pl.when(pl.col("age_diff") <= 3)
          .then(pl.lit("<=3"))
        .when((pl.col("age_diff") > 3) & (pl.col("age_diff") <= 5))
          .then(pl.lit("3-6"))
        .otherwise(pl.lit(">5"))
        .alias("diff_bucket")
    )
    .group_by("diff_bucket")
    .agg(pl.count().alias("num_customers"))
    .sort("diff_bucket")
)

print("=== Phân bố độ lệch tuổi ===")
print(diff_bucket_stats)


=== Phân bố độ lệch tuổi ===
shape: (3, 2)
┌─────────────┬───────────────┐
│ diff_bucket ┆ num_customers │
│ ---         ┆ ---           │
│ str         ┆ u32           │
╞═════════════╪═══════════════╡
│ 3-6         ┆ 18497         │
│ <=3         ┆ 234786        │
│ >5          ┆ 17661         │
└─────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_500\3055082791.py:16: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [94]:
customer_with_age

customer_id,first_date_buy_step1,age_by_step1,first_date_buy_age_group_0_3M,age_by_age_group,last_date_buy_milk4mom,age_by_milk4mom,age_diff,age_final
i32,date,f64,date,f64,date,f64,f64,f64
4748251,null,null,null,null,null,null,null,null
6704093,null,null,2024-02-03,13.6,null,null,null,13.6
8092223,null,null,null,null,null,null,null,null
7105333,null,null,null,null,null,null,null,null
2979082,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…
6878538,null,null,null,null,null,null,null,null
7844605,null,null,null,null,null,null,null,null
114947,null,null,null,null,null,null,null,null


In [95]:
customer_with_age.write_parquet(OUTPUT_PATH)
print("Đã lưu:", OUTPUT_PATH)


Đã lưu: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\customer_age_features_2401_2501.parquet


### **Đặc trưng:** Có mua đa dạng brand không. 
Cũng chỉ xét "Sữa", "Tã"

In [ ]:
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"
df_item = pl.read_parquet(path_item)

print(df_item.columns)


['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'item_type', 'gender_target_final', 'description_final', 'brand_final', 'age_group_final', 'price_segment']


In [ ]:
# df_trans: transaction (đã load từng chunk rồi concat hoặc xử lý chunk-wise)
# df_item: item đã có category_l1, brand_final

df = (
    df_trans
    .select(["item_id", "customer_id"])
    .join(
        df_item.select(["item_id", "category_l1", "brand_final"]),
        on="item_id",
        how="left"
    )
    .filter(
        pl.col("category_l1").is_in(["Sữa", "Tã"]) &
        pl.col("brand_final").is_not_null()
    )
)

brand_agg = (
    df
    .group_by(["customer_id", "category_l1"])
    .agg([
        pl.count().alias("num_transactions"),
        pl.col("brand_final").n_unique().alias("num_unique_brands"),
    ])
    .with_columns(
        (pl.col("num_unique_brands") / pl.col("num_transactions"))
        .alias("brand_diversity_ratio")
    )
)


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\2008330972.py:22: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_transactions"),


In [ ]:
def cluster_brand_diversity(pdf):
    X = pdf[["brand_diversity_ratio"]].to_numpy()

    kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X)

    pdf["cluster_raw"] = labels

    # Sắp cluster theo mean ratio
    means = (
        pdf.groupby("cluster_raw")["brand_diversity_ratio"]
        .mean()
        .sort_values()
    )

    mapping = {cluster: i for i, cluster in enumerate(means.index)}
    pdf["brand_segment"] = pdf["cluster_raw"].map(mapping)

    return pdf.drop(columns=["cluster_raw"])


In [ ]:
# Tách theo category_l1
milk_pdf = brand_agg.filter(pl.col("category_l1") == "Sữa").to_pandas()
diaper_pdf = brand_agg.filter(pl.col("category_l1") == "Tã").to_pandas()

milk_clustered = cluster_brand_diversity(milk_pdf)
diaper_clustered = cluster_brand_diversity(diaper_pdf)

brand_segment_df = pl.concat([
    pl.from_pandas(milk_clustered),
    pl.from_pandas(diaper_clustered)
])


In [ ]:
print(
    brand_segment_df
    .group_by(["category_l1", "brand_segment"])
    .agg(pl.count().alias("num_customers"))
    .sort(["category_l1", "brand_segment"])
)


shape: (6, 3)
┌─────────────┬───────────────┬───────────────┐
│ category_l1 ┆ brand_segment ┆ num_customers │
│ ---         ┆ ---           ┆ ---           │
│ str         ┆ i64           ┆ u32           │
╞═════════════╪═══════════════╪═══════════════╡
│ Sữa         ┆ 0             ┆ 227926        │
│ Sữa         ┆ 1             ┆ 233241        │
│ Sữa         ┆ 2             ┆ 435776        │
│ Tã          ┆ 0             ┆ 227775        │
│ Tã          ┆ 1             ┆ 163506        │
│ Tã          ┆ 2             ┆ 375697        │
└─────────────┴───────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\3827601304.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [ ]:
print(
    brand_segment_df
    .group_by(["category_l1", "brand_segment"])
    .agg([
        pl.col("brand_diversity_ratio").min().alias("min"),
        pl.col("brand_diversity_ratio").mean().alias("mean"),
        pl.col("brand_diversity_ratio").median().alias("median"),
        pl.col("brand_diversity_ratio").max().alias("max"),
    ])
    .sort(["category_l1", "brand_segment"])
)


shape: (6, 6)
┌─────────────┬───────────────┬──────────┬──────────┬──────────┬──────────┐
│ category_l1 ┆ brand_segment ┆ min      ┆ mean     ┆ median   ┆ max      │
│ ---         ┆ ---           ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str         ┆ i64           ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞═════════════╪═══════════════╪══════════╪══════════╪══════════╪══════════╡
│ Sữa         ┆ 0             ┆ 0.003984 ┆ 0.164849 ┆ 0.166667 ┆ 0.315789 │
│ Sữa         ┆ 1             ┆ 0.318182 ┆ 0.470452 ┆ 0.5      ┆ 0.727273 │
│ Sữa         ┆ 2             ┆ 0.733333 ┆ 0.995604 ┆ 1.0      ┆ 1.0      │
│ Tã          ┆ 0             ┆ 0.004184 ┆ 0.209033 ┆ 0.2      ┆ 0.363636 │
│ Tã          ┆ 1             ┆ 0.368421 ┆ 0.524987 ┆ 0.5      ┆ 0.75     │
│ Tã          ┆ 2             ┆ 0.777778 ┆ 0.999263 ┆ 1.0      ┆ 1.0      │
└─────────────┴───────────────┴──────────┴──────────┴──────────┴──────────┘


In [ ]:
for seg in [0, 1, 2]:
    print(f"\n=== Random khách segment {seg} ===")
    print(
        brand_segment_df
        .filter(pl.col("brand_segment") == seg)
        .sample(n=10, seed=42)
        .select([
            "customer_id",
            "category_l1",
            "num_transactions",
            "num_unique_brands",
            "brand_diversity_ratio"
        ])
    )



=== Random khách segment 0 ===
shape: (10, 5)
┌─────────────┬─────────────┬──────────────────┬───────────────────┬───────────────────────┐
│ customer_id ┆ category_l1 ┆ num_transactions ┆ num_unique_brands ┆ brand_diversity_ratio │
│ ---         ┆ ---         ┆ ---              ┆ ---               ┆ ---                   │
│ i32         ┆ str         ┆ u32              ┆ u32               ┆ f64                   │
╞═════════════╪═════════════╪══════════════════╪═══════════════════╪═══════════════════════╡
│ 1570091     ┆ Tã          ┆ 7                ┆ 2                 ┆ 0.285714              │
│ 5730926     ┆ Sữa         ┆ 10               ┆ 2                 ┆ 0.2                   │
│ 3801854     ┆ Tã          ┆ 11               ┆ 4                 ┆ 0.363636              │
│ 6102929     ┆ Tã          ┆ 8                ┆ 1                 ┆ 0.125                 │
│ 6497599     ┆ Tã          ┆ 6                ┆ 2                 ┆ 0.333333              │
│ 3072179     ┆ Tã     

In [ ]:

# Đường dẫn output
OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "brand_segment.parquet")

# Đảm bảo thư mục tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Chỉ giữ các cột cần thiết và lưu
(
    brand_segment_df
    .select(["category_l1", "brand_segment"])
    .write_parquet(OUTPUT_PATH)
)

print("✅ Đã lưu brand_segment thành công tại:")
print(OUTPUT_PATH)

✅ Đã lưu brand_segment thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\brand_segment.parquet


### **Đặc trưng về hành vi mua hàng:** Xét hành vi mua hàng dựa trên độ tuổi của bé
Lấy độ đa dạng trong category_1, lấy top K sản phẩm tại mỗi loại để xem người dùng có bé trong độ tuổi đó thì thường mua những sản phẩm nào

### **Đặc trưng về hành vi mua hàng:** Thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc (Cũng xét theo ngày)

In [2]:
from itertools import permutations


In [5]:
TX_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\preprocessing data"
TX_PATTERN = "sale_pers.purchase_history_daily_chunk_{}.parquet"

TMP_DIR = r"D:\tmp_item_cooc_chunks"
OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\item_co_items_202401_202501.parquet"

os.makedirs(TMP_DIR, exist_ok=True)

In [6]:
# TIME WINDOW
START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2025, 1, 31)

# PARAMETERS
MAX_BASKET_SIZE = 10    # bắt buộc để sống
MIN_COOC = 600

In [7]:
def generate_pairs(items):
    """
    items: list[item_id], đã unique
    return: list[(a, b)] có hướng
    """
    if len(items) < 2 or len(items) > MAX_BASKET_SIZE:
        return []
    return list(permutations(items, 2))


In [8]:
for i in range(80):  # chunk_0 → chunk_79
    tx_path = os.path.join(TX_DIR, TX_PATTERN.format(i))
    if not os.path.exists(tx_path):
        continue

    print(f"\n=== Processing chunk {i} ===")

    # --- Read minimal columns + filter time ---
    tx = (
        pl.read_parquet(
            tx_path,
            columns=["customer_id", "item_id", "created_date"]
        )
        .filter(
            (pl.col("created_date") >= START_DATE) &
            (pl.col("created_date") <= END_DATE)
        )
    )

    if tx.is_empty():
        print("  -> No data after time filter")
        continue

    print("  Transactions:", tx.height)

    # --- Basket theo (customer, day) ---
    baskets = (
        tx
        .group_by(["customer_id", "created_date"])
        .agg(pl.col("item_id").unique().alias("items"))
        .filter(
            (pl.col("items").list.len() >= 2) &
            (pl.col("items").list.len() <= MAX_BASKET_SIZE)
        )
    )

    print("  Valid baskets:", baskets.height)

    if baskets.is_empty():
        print("  -> No valid baskets")
        continue

    # --- Generate item pairs ---
    pairs = (
        baskets
        .with_columns(
            pl.col("items")
            .map_elements(
                generate_pairs,
                return_dtype=pl.List(pl.List(pl.Utf8))
            )
            .alias("pairs")
        )
        .explode("pairs")
        .with_columns([
            pl.col("pairs").list.get(0).alias("item_id_a"),
            pl.col("pairs").list.get(1).alias("item_id_b"),
        ])
        .select(["item_id_a", "item_id_b"])
    )

    print("  Raw pairs:", pairs.height)

    if pairs.is_empty():
        print("  -> No pairs generated")
        continue

    # --- Count co-occurrence in this chunk ---
    pair_count = (
        pairs
        .group_by(["item_id_a", "item_id_b"])
        .agg(pl.len().alias("cooc_count"))
    )

    print("  Unique pairs in chunk:", pair_count.height)

    # --- WRITE TO DISK (CỰC KỲ QUAN TRỌNG) ---
    out_chunk = os.path.join(TMP_DIR, f"pair_chunk_{i}.parquet")
    pair_count.write_parquet(out_chunk)

    print("  Saved:", out_chunk)



=== Processing chunk 0 ===
  Transactions: 487850
  Valid baskets: 103124
  Raw pairs: 1054186
  Unique pairs in chunk: 620584
  Saved: D:\tmp_item_cooc_chunks\pair_chunk_0.parquet

=== Processing chunk 1 ===
  Transactions: 487850
  Valid baskets: 102046
  Raw pairs: 1005342
  Unique pairs in chunk: 580896
  Saved: D:\tmp_item_cooc_chunks\pair_chunk_1.parquet

=== Processing chunk 2 ===
  Transactions: 487850
  Valid baskets: 100691
  Raw pairs: 1071660
  Unique pairs in chunk: 606284
  Saved: D:\tmp_item_cooc_chunks\pair_chunk_2.parquet

=== Processing chunk 3 ===
  Transactions: 487850
  Valid baskets: 100834
  Raw pairs: 1029268
  Unique pairs in chunk: 591808
  Saved: D:\tmp_item_cooc_chunks\pair_chunk_3.parquet

=== Processing chunk 4 ===
  Transactions: 487850
  Valid baskets: 102312
  Raw pairs: 1128566
  Unique pairs in chunk: 611706
  Saved: D:\tmp_item_cooc_chunks\pair_chunk_4.parquet

=== Processing chunk 5 ===
  Transactions: 487850
  Valid baskets: 99331
  Raw pairs: 108

In [9]:
print("\n=== PHASE 3: Aggregating all chunk files ===")

pair_files = [
    os.path.join(TMP_DIR, f)
    for f in os.listdir(TMP_DIR)
    if f.endswith(".parquet")
]

print("Num tmp files:", len(pair_files))

pair_all = pl.concat(
    [pl.read_parquet(f) for f in pair_files],
    how="vertical"
)

print("Total rows before final agg:", pair_all.height)

pair_strong = (
    pair_all
    .group_by(["item_id_a", "item_id_b"])
    .agg(pl.col("cooc_count").sum())
    .filter(pl.col("cooc_count") >= MIN_COOC)
    .sort("cooc_count", descending=True)
)

print("Strong pairs (>=600):", pair_strong.height)
print(pair_strong.head(10))



=== PHASE 3: Aggregating all chunk files ===
Num tmp files: 80
Total rows before final agg: 45173278
Strong pairs (>=600): 8024
shape: (10, 3)
┌───────────────┬───────────────┬────────────┐
│ item_id_a     ┆ item_id_b     ┆ cooc_count │
│ ---           ┆ ---           ┆ ---        │
│ str           ┆ str           ┆ u32        │
╞═══════════════╪═══════════════╪════════════╡
│ 2803000000013 ┆ 2803000000011 ┆ 57191      │
│ 2803000000011 ┆ 2803000000013 ┆ 57191      │
│ 2803000000012 ┆ 2803000000013 ┆ 39380      │
│ 2803000000013 ┆ 2803000000012 ┆ 39380      │
│ 2803000000012 ┆ 2803000000011 ┆ 38668      │
│ 2803000000011 ┆ 2803000000012 ┆ 38668      │
│ 2803000000010 ┆ 2803000000012 ┆ 29799      │
│ 2803000000012 ┆ 2803000000010 ┆ 29799      │
│ 2803000000013 ┆ 2803000000010 ┆ 23959      │
│ 2803000000010 ┆ 2803000000013 ┆ 23959      │
└───────────────┴───────────────┴────────────┘


In [ ]:
# ========== check ==================

In [16]:
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\sale_pers.item_chunk_0.parquet"

item_meta = pl.read_parquet(
    ITEM_PATH,
    columns=[
        "item_id",
        "category_l1",
        "category",     # nếu cột này tồn tại
        "brand_final"
    ]
)

print("Item meta shape:", item_meta.shape)
print(item_meta.head())


Item meta shape: (27323, 4)
shape: (5, 4)
┌───────────────┬────────────────┬───────────────────┬──────────────────┐
│ item_id       ┆ category_l1    ┆ category          ┆ brand_final      │
│ ---           ┆ ---            ┆ ---               ┆ ---              │
│ str           ┆ str            ┆ str               ┆ str              │
╞═══════════════╪════════════════╪═══════════════════╪══════════════════╡
│ 0502020000004 ┆ Babycare       ┆ Núm ty Dr Brown   ┆ Dr.Brown's       │
│ 0010290040150 ┆ Thời trang     ┆ Bộ quần áo bé gái ┆ Con Cưng         │
│ 0008010000015 ┆ Đồ chơi & Sách ┆ Gặm nướu khác     ┆ Thương hiệu khác │
│ 0020010000094 ┆ Tã             ┆ Merries_Sơ Sinh   ┆ Merries          │
│ 0020010000098 ┆ Tã             ┆ Merries_Tã Quần   ┆ Merries          │
└───────────────┴────────────────┴───────────────────┴──────────────────┘


In [18]:
pair_with_a = pair_strong.join(
    item_meta.rename({
        "item_id": "item_id_a",
        "category_l1": "a_category_l1",
        "category": "a_category",
        "brand_final": "a_brand",
    }),
    on="item_id_a",
    how="left"
)


In [19]:
pair_with_ab = pair_with_a.join(
    item_meta.rename({
        "item_id": "item_id_b",
        "category_l1": "b_category_l1",
        "category": "b_category",
        "brand_final": "b_brand",
    }),
    on="item_id_b",
    how="left"
)


In [20]:
pair_with_ab.select([
    "item_id_a",
    "a_category_l1", "a_category", "a_brand",
    "item_id_b",
    "b_category_l1", "b_category", "b_brand",
    "cooc_count"
]).head(20)


item_id_a,a_category_l1,a_category,a_brand,item_id_b,b_category_l1,b_category,b_brand,cooc_count
str,str,str,str,str,str,str,str,u32
"""2803000000013""","""Thực phẩm cho bé""","""Hoff""","""HOFF""","""2803000000011""","""Thực phẩm cho bé""","""Hoff""","""HOFF""",57191
"""2803000000011""","""Thực phẩm cho bé""","""Hoff""","""HOFF""","""2803000000013""","""Thực phẩm cho bé""","""Hoff""","""HOFF""",57191
"""2803000000012""","""Thực phẩm cho bé""","""Hoff""","""HOFF""","""2803000000013""","""Thực phẩm cho bé""","""Hoff""","""HOFF""",39380
"""2803000000013""","""Thực phẩm cho bé""","""Hoff""","""HOFF""","""2803000000012""","""Thực phẩm cho bé""","""Hoff""","""HOFF""",39380
"""2803000000012""","""Thực phẩm cho bé""","""Hoff""","""HOFF""","""2803000000011""","""Thực phẩm cho bé""","""Hoff""","""HOFF""",38668
…,…,…,…,…,…,…,…,…
"""0029130000030""","""Thực phẩm cho bé""","""Vinamilk""","""Ridielac Gold""","""0029130000029""","""Thực phẩm cho bé""","""Vinamilk""","""Ridielac Gold""",20526
"""0029250010002""","""Thực phẩm cho bé""","""Ivenet""","""Ivenet""","""0029250010001""","""Thực phẩm cho bé""","""Ivenet""","""Ivenet""",20362
"""0029250010001""","""Thực phẩm cho bé""","""Ivenet""","""Ivenet""","""0029250010002""","""Thực phẩm cho bé""","""Ivenet""","""Ivenet""",20362


In [25]:
pair_with_ab.select([
    "item_id_a",
    "a_category_l1", "a_brand",
    "item_id_b",
    "b_category_l1", "b_brand",
    "cooc_count"
]).sample(n=10, shuffle=True)


item_id_a,a_category_l1,a_brand,item_id_b,b_category_l1,b_brand,cooc_count
str,str,str,str,str,str,u32
"""0007150000143""","""Vệ sinh""","""BeeVn""","""4603024000001""","""Hóa mỹ phẩm cho bé""","""Animo""",743
"""6768000000003""","""Tã""","""Takato""","""5950000000001""","""Babycare""","""Dr Papie""",924
"""4396000000003""","""Thực phẩm cho bé""","""Grinny""","""7115000000005""","""Thực phẩm cho bé""","""Grinny""",2541
"""6697000000004""","""Sữa nước""","""GrowPLUS+""","""2024000000011""","""Sữa nước""","""PediaSure""",611
"""0020020000173""","""Thực phẩm cho bé""","""Sài Gòn Food""","""0020020000202""","""Thực phẩm cho bé""","""Sài Gòn Food""",700
"""2609000000002""","""Thực phẩm cho bé""","""Playmore""","""2707000000001""","""Thực phẩm cho bé""","""Chupa chups""",1156
"""0029250010024""","""Thực phẩm cho bé""","""Ivenet""","""0029250010025""","""Thực phẩm cho bé""","""Ivenet""",1528
"""0020020000104""","""Thực phẩm cho bé""","""Beanstalk""","""0020020000105""","""Thực phẩm cho bé""","""Beanstalk""",1688
"""3880000000002""","""Thực phẩm cho bé""","""HOFF""","""5503000000004""","""Thực phẩm cho bé""","""Bebedang""",686


In [ ]:
# ============== done check =====================

In [10]:
final_output = (
    pair_strong
    .sort(["item_id_a", "cooc_count"], descending=[False, True])
    .group_by("item_id_a")
    .agg(
        pl.col("item_id_b").alias("co_items")
    )
    .sort("item_id_a")
)

print("\n=== FINAL OUTPUT SAMPLE ===")
print(final_output.head(10))
print("Num item a:", final_output.height)



=== FINAL OUTPUT SAMPLE ===
shape: (10, 2)
┌───────────────┬─────────────────────────────────┐
│ item_id_a     ┆ co_items                        │
│ ---           ┆ ---                             │
│ str           ┆ list[str]                       │
╞═══════════════╪═════════════════════════════════╡
│ 0000280000138 ┆ ["0009200000049"]               │
│ 0006020000278 ┆ ["0020010000210"]               │
│ 0006020000279 ┆ ["0020010000210"]               │
│ 0006020000281 ┆ ["0020010000210"]               │
│ 0006040000270 ┆ ["0006040000273"]               │
│ 0006040000273 ┆ ["0006040000270"]               │
│ 0006040000318 ┆ ["0007010000887"]               │
│ 0006040000478 ┆ ["0055000000002"]               │
│ 0006040000479 ┆ ["0055000000002", "00420000000… │
│ 0007010000290 ┆ ["4603024000001", "63820000000… │
└───────────────┴─────────────────────────────────┘
Num item a: 773


In [11]:
preview = final_output.with_columns(
    pl.col("co_items").list.join(",").alias("co_items_str")
)

print(preview.head(10))


shape: (10, 3)
┌───────────────┬─────────────────────────────────┬─────────────────────────────┐
│ item_id_a     ┆ co_items                        ┆ co_items_str                │
│ ---           ┆ ---                             ┆ ---                         │
│ str           ┆ list[str]                       ┆ str                         │
╞═══════════════╪═════════════════════════════════╪═════════════════════════════╡
│ 0000280000138 ┆ ["0009200000049"]               ┆ 0009200000049               │
│ 0006020000278 ┆ ["0020010000210"]               ┆ 0020010000210               │
│ 0006020000279 ┆ ["0020010000210"]               ┆ 0020010000210               │
│ 0006020000281 ┆ ["0020010000210"]               ┆ 0020010000210               │
│ 0006040000270 ┆ ["0006040000273"]               ┆ 0006040000273               │
│ 0006040000273 ┆ ["0006040000270"]               ┆ 0006040000270               │
│ 0006040000318 ┆ ["0007010000887"]               ┆ 0007010000887               │
│

In [27]:
preview.sample(n=10, shuffle=True)


item_id_a,co_items,co_items_str
str,list[str],str
"""4051000000002""","[""1512000000004"", ""4048000000008""]","""1512000000004,4048000000008"""
"""3773000000003""","[""1512000000004"", ""4690000000001""]","""1512000000004,4690000000001"""
"""5506000000002""","[""5506000000003"", ""5506000000004"", … ""5537000000007""]","""5506000000003,5506000000004,55…"
"""2610000000008""","[""2610000000004""]","""2610000000004"""
"""2609000000001""","[""2609000000002"", ""2707000000001"", ""7161000000001""]","""2609000000002,2707000000001,71…"
"""0029250010001""","[""0029250010003"", ""0029250010002"", … ""6501000000007""]","""0029250010003,0029250010002,28…"
"""0055000000002""","[""0042000000005"", ""0006040000479"", ""0006040000478""]","""0042000000005,0006040000479,00…"
"""1438000000002""","[""1438000000001""]","""1438000000001"""
"""5503000000002""","[""5503000000003"", ""5503000000004"", … ""1308000000003""]","""5503000000003,5503000000004,55…"


In [28]:
final_output

item_id_a,co_items
str,list[str]
"""0000280000138""","[""0009200000049""]"
"""0006020000278""","[""0020010000210""]"
"""0006020000279""","[""0020010000210""]"
"""0006020000281""","[""0020010000210""]"
"""0006040000270""","[""0006040000273""]"
…,…
"""7228000000016""","[""7228000000015"", ""7228000000023""]"
"""7228000000017""","[""7228000000023"", ""1952000000016"", ""0068000000160""]"
"""7228000000018""","[""0068000000159"", ""7228000000024"", ""1952000000016""]"


In [29]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
final_output.write_parquet(OUTPUT_PATH)

print("\nSaved final output to:")
print(OUTPUT_PATH)



Saved final output to:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-3\Feature engineering\item_co_items_202401_202501.parquet


### Discount user: Người dùng đó có mua hàng discount hay k. Tính dựa trên số lần mua trong tháng
Cũng có thể sẽ là gom cụm: mua ít, mua vừa, mua nhiều

### Đếm số lượng mặt hàng được bán ra (Mặt hàng được bán ra càng nhiều thì khả năng người ta mua hàng sẽ càng cao ) 
(Hệ số phổ biến của sản phẩm)

chuẩn hóa theo từng category_l1, và chỉ giữ TOP 10 item phổ biến nhất trong mỗi category_l1.

In [1]:

# =========================
# 1) TÍNH total_sold THEO item_id (gộp 20 chunk)
# =========================
TRANS_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
trans_files = sorted(glob.glob(f"{TRANS_PATH}\\sale_pers.purchase_history_daily_chunk_*.parquet"))

item_sales = []

for file in trans_files:
    print("Processing:", file)

    df = (
        pl.read_parquet(file)
        .select(["item_id", "quantity"])
        .group_by("item_id")
        .agg(pl.col("quantity").sum().alias("total_sold"))
    )
    item_sales.append(df)

item_sales_all = (
    pl.concat(item_sales, how="vertical_relaxed")
    .group_by("item_id")
    .agg(pl.col("total_sold").sum().alias("total_sold"))
)

print("item_sales_all shape:", item_sales_all.shape)

# =========================
# 2) JOIN LẤY category_l1
# =========================
# df_item: item dataframe của bạn (đã load sẵn) phải có ["item_id","category_l1"]
item_basic = df_item.select(["item_id", "category_l1"])

item_sales_cat = (
    item_sales_all
    .join(item_basic, on="item_id", how="left")
    .filter(pl.col("category_l1").is_not_null())
)

print("item_sales_cat shape:", item_sales_cat.shape)

# =========================
# 3) TOP 10 item_id THEO MỖI category_l1
# =========================
top10_by_cat = (
    item_sales_cat
    .sort(["category_l1", "total_sold"], descending=[False, True])
    .group_by("category_l1")
    .head(10)
    .sort(["category_l1", "total_sold"], descending=[False, True])
)

print("\n✅ Top 10 item phổ biến nhất theo từng category_l1 (mẫu):")
print(top10_by_cat.head(30))

# =========================
# 4) CHECK: in ra 1 category bất kỳ để bạn nhìn rõ 10 item_id
# =========================
sample_cat = top10_by_cat.select("category_l1").unique().head(1).item()
print(f"\n=== CHECK TOP 10 của category_l1 = {sample_cat} ===")
print(
    top10_by_cat
    .filter(pl.col("category_l1") == sample_cat)
    .select(["category_l1", "item_id", "total_sold"])
)


NameError: name 'glob' is not defined

In [ ]:
top10_by_cat.head(50)

category_l1,item_id,total_sold
str,str,i32
"""Babycare""","""5950000000001""",203388
"""Babycare""","""0203000000004""",128333
"""Babycare""","""0007150000144""",123655
"""Babycare""","""0007150000031""",80181
"""Babycare""","""6498000000005""",60252
…,…,…
"""Sữa""","""2483000000004""",115008
"""Sữa""","""4355000000001""",99082
"""Sữa""","""6488000000001""",97755


In [ ]:
# =========================
# SAVE top10_by_cat
# =========================

OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = f"{OUTPUT_DIR}\\top10_by_cat.parquet"

top10_by_cat.write_parquet(OUTPUT_PATH)

print("✅ Đã lưu top10_by_cat xuống:")
print(OUTPUT_PATH)
print("Shape:", top10_by_cat.shape)


✅ Đã lưu top10_by_cat xuống:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\top10_by_cat.parquet
Shape: (140, 3)


**Xét Đếm số lượng mặt hàng được bán ra (Mặt hàng được bán ra càng nhiều thì khả năng người ta mua hàng sẽ càng cao ) (Hệ số phổ biến của sản phẩm), chuẩn hóa theo từng category_l1, và chỉ giữ TOP 10 item phổ biến nhất trong mỗi category_l1. Nhưng đếm theo tháng**

Tức là kiểu như tháng nào người dùng sẽ thường mua sản phẩm nào. Để sau này ví dụ thầy cho dữ liệu tháng 1/2025 thì có thể dùng những sản phẩm phổ biến của tháng 11/2024 để gợi ý

In [ ]:

TRANS_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
ITEM_PATH  = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

trans_files = sorted(
    glob.glob(f"{TRANS_PATH}/sale_pers.purchase_history_daily_chunk_*.parquet")
)

item = (
    pl.read_parquet(ITEM_PATH)
    .select(["item_id", "category_l1"])
)

In [ ]:
monthly_sales_chunks = []

for file in trans_files:
    print("Processing:", os.path.basename(file))

    df = (
        pl.read_parquet(file)
        .select(["item_id", "quantity", "created_date"])
        .with_columns(
            pl.col("created_date")
            .cast(pl.Date)
            .dt.strftime("%Y-%m")
            .alias("month")
        )
        .join(item, on="item_id", how="left")
        .filter(pl.col("category_l1").is_not_null())
        .group_by(["month", "category_l1", "item_id"])
        .agg(
            pl.col("quantity").sum().alias("total_sold")
        )
    )

    monthly_sales_chunks.append(df)


Processing: sale_pers.purchase_history_daily_chunk_0.parquet
Processing: sale_pers.purchase_history_daily_chunk_1.parquet
Processing: sale_pers.purchase_history_daily_chunk_10.parquet
Processing: sale_pers.purchase_history_daily_chunk_11.parquet
Processing: sale_pers.purchase_history_daily_chunk_12.parquet
Processing: sale_pers.purchase_history_daily_chunk_13.parquet
Processing: sale_pers.purchase_history_daily_chunk_14.parquet
Processing: sale_pers.purchase_history_daily_chunk_15.parquet
Processing: sale_pers.purchase_history_daily_chunk_16.parquet
Processing: sale_pers.purchase_history_daily_chunk_17.parquet
Processing: sale_pers.purchase_history_daily_chunk_18.parquet
Processing: sale_pers.purchase_history_daily_chunk_19.parquet
Processing: sale_pers.purchase_history_daily_chunk_2.parquet
Processing: sale_pers.purchase_history_daily_chunk_3.parquet
Processing: sale_pers.purchase_history_daily_chunk_4.parquet
Processing: sale_pers.purchase_history_daily_chunk_5.parquet
Processing: sa

In [ ]:
monthly_sales_all = (
    pl.concat(monthly_sales_chunks, how="vertical_relaxed")
    .group_by(["month", "category_l1", "item_id"])
    .agg(
        pl.col("total_sold").sum().alias("total_sold")
    )
)

print("monthly_sales_all shape:", monthly_sales_all.shape)


monthly_sales_all shape: (153563, 4)


In [ ]:
top10_by_cat_month = (
    monthly_sales_all
    .with_columns(
        pl.col("total_sold")
        .rank(method="dense", descending=True)
        .over(["month", "category_l1"])
        .alias("rank")
    )
    .filter(pl.col("rank") <= 10)
    .sort(["month", "category_l1", "rank"])
)

print("top10_by_cat_month shape:", top10_by_cat_month.shape)


top10_by_cat_month shape: (1691, 5)


In [ ]:
print(
    top10_by_cat_month
    .filter(pl.col("month") == "2024-11")
    .head(50)
)


shape: (50, 5)
┌─────────┬─────────────┬───────────────┬────────────┬──────┐
│ month   ┆ category_l1 ┆ item_id       ┆ total_sold ┆ rank │
│ ---     ┆ ---         ┆ ---           ┆ ---        ┆ ---  │
│ str     ┆ str         ┆ str           ┆ i32        ┆ u32  │
╞═════════╪═════════════╪═══════════════╪════════════╪══════╡
│ 2024-11 ┆ Babycare    ┆ 5950000000001 ┆ 16759      ┆ 1    │
│ 2024-11 ┆ Babycare    ┆ 0203000000004 ┆ 10882      ┆ 2    │
│ 2024-11 ┆ Babycare    ┆ 0007150000144 ┆ 10481      ┆ 3    │
│ 2024-11 ┆ Babycare    ┆ 0007150000031 ┆ 6373       ┆ 4    │
│ 2024-11 ┆ Babycare    ┆ 6498000000005 ┆ 5682       ┆ 5    │
│ …       ┆ …           ┆ …             ┆ …          ┆ …    │
│ 2024-11 ┆ Sữa         ┆ 3774000000003 ┆ 13280      ┆ 5    │
│ 2024-11 ┆ Sữa         ┆ 4950000000001 ┆ 12142      ┆ 6    │
│ 2024-11 ┆ Sữa         ┆ 2483000000004 ┆ 9208       ┆ 7    │
│ 2024-11 ┆ Sữa         ┆ 2482000000004 ┆ 8949       ┆ 8    │
│ 2024-11 ┆ Sữa         ┆ 2578000000002 ┆ 8463       ┆ 

In [ ]:
num_months = top10_by_cat_month.select("month").n_unique()

print(f"Số lượng tháng được lọc ra: {num_months}")


Số lượng tháng được lọc ra: 12


In [ ]:
check = (
    top10_by_cat_month
    .group_by(["month", "category_l1"])
    .agg(pl.count().alias("num_items"))
    .filter(pl.col("num_items") != 10)
)

print("Các group KHÔNG đủ 10 item (nếu có):")
print(check)


Các group KHÔNG đủ 10 item (nếu có):
shape: (11, 3)
┌─────────┬────────────────────────┬───────────┐
│ month   ┆ category_l1            ┆ num_items │
│ ---     ┆ ---                    ┆ ---       │
│ str     ┆ str                    ┆ u32       │
╞═════════╪════════════════════════╪═══════════╡
│ 2024-12 ┆ Thực phẩm cho bé       ┆ 11        │
│ 2024-08 ┆ Thời trang             ┆ 11        │
│ 2024-09 ┆ Tã                     ┆ 11        │
│ 2024-07 ┆ Thực phẩm cho gia đình ┆ 11        │
│ 2024-03 ┆ Thực phẩm cho gia đình ┆ 11        │
│ …       ┆ …                      ┆ …         │
│ 2024-06 ┆ Hóa mỹ phẩm gia đình   ┆ 11        │
│ 2024-11 ┆ Phụ kiện               ┆ 11        │
│ 2024-10 ┆ Tã                     ┆ 11        │
│ 2024-10 ┆ Phụ kiện               ┆ 11        │
│ 2024-01 ┆ Phụ kiện               ┆ 11        │
└─────────┴────────────────────────┴───────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\1899442887.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_items"))


In [ ]:
OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = f"{OUTPUT_DIR}\\top10_by_cat_month.parquet"

top10_by_cat_month.write_parquet(OUTPUT_PATH)

print("✅ Đã lưu top10_by_cat_month")
print("Path:", OUTPUT_PATH)


✅ Đã lưu top10_by_cat_month
Path: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\top10_by_cat_month.parquet


### Lần cuối cùng mua hàng của khách hàng

In [ ]:
# Như tạo r biến: Recency

### =========================================================================================

In [ ]:
pl.Config.set_tbl_rows(100)

polars.config.Config